# ARG Dashboard V2 BACI Pipeline

This notebook is the source of truth for Argentina Dashboard V2 intermediate files. It mirrors the Ecuador reference workflow in `context/dashboard_context_bundle.zip`, adapted to Argentina and CEPII BACI HS92 data.

It produces the runtime files required by Streamlit:

- `data/intermediate/complexity_arg_2024.csv`
- `data/intermediate/opportunity_metrics_hs4_arg.csv`
- `data/intermediate/hs92_attributes.csv`
- `data/output/anchors_proximity_percentile.csv`

It also writes reproducibility intermediates used to audit the pipeline:

- `data/intermediate/countries.csv`
- `data/intermediate/trade_hs4_baci_2020_2024.csv`
- `data/intermediate/accessible_market_*`
- `data/intermediate/complexity_calculations.csv`
- `data/intermediate/proximity.csv`


In [1]:
from pathlib import Path
import math
import os
import numpy as np
import pandas as pd
from ecomplexity import ecomplexity, proximity

FOCUS_ISO = 'ARG'
FOCUS_COUNTRY_NAME = 'Argentina'
FOCUS_COUNTRY_SLUG = 'argentina'
BACI_VERSION = 'V202601'
YEARS = [2020, 2021, 2022, 2023, 2024]
ACCESSIBLE_EXPORT_THRESHOLD_USD = 100_000_000.0

project_root = Path.cwd()
if project_root.name == 'code':
    project_root = project_root.parent
elif (project_root / 'ARG_Dashboard_V2' / 'data' / 'input').exists():
    project_root = project_root / 'ARG_Dashboard_V2'
elif not (project_root / 'data' / 'input').exists():
    for parent in project_root.parents:
        candidate = parent / 'ARG_Dashboard_V2'
        if (candidate / 'data' / 'input').exists():
            project_root = candidate
            break
os.chdir(project_root)

INPUT = project_root / 'data' / 'input'
INTERMEDIATE = project_root / 'data' / 'intermediate'
OUTPUT = project_root / 'data' / 'output'
INTERMEDIATE.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

print(project_root)
print('Using BACI only for trade flows')

/Users/joc0445/Library/CloudStorage/OneDrive-HarvardUniversity/VS/Cordoba/ARG_Dashboard_V2/GL_argentina_opportunities
Using BACI only for trade flows


## 1. Countries

Keep the dashboard country universe from `rankings.csv`, year 2024, restricted to countries marked `in_rankings` when that column exists.

In [2]:
rankings = pd.read_csv(INPUT / 'rankings.csv')
countries = rankings[pd.to_numeric(rankings['year'], errors='coerce').eq(2024)].copy()
if 'in_rankings' in countries.columns:
    countries = countries[countries['in_rankings'].astype(bool)].copy()
countries = countries.rename(columns={'country_iso3_code': 'iso3'})
countries['iso3'] = countries['iso3'].astype(str).str.upper().str.strip().str[:3]
countries = countries[countries['iso3'].str.len().eq(3)][['iso3']].drop_duplicates().sort_values('iso3')
countries.to_csv(INTERMEDIATE / 'countries.csv', index=False)
valid_countries = set(countries['iso3'])
print(f'countries: {len(valid_countries):,}')
countries.head()

countries: 145


,iso3
29,AFG
119,AGO
59,ALB
3838,ARE
179,ARG


## 2. Build BACI HS4 Bilateral Trade Base

This converts BACI HS6 rows (`t`, `i`, `j`, `k`, `v`) to an HS4 bilateral panel in USD. BACI `v` is reported in thousand USD, so it is multiplied by 1,000.

In [3]:
baci_dir_candidates = [
    INPUT / 'BACI_HS92_V202601',
    project_root.parent / 'data' / 'input' / 'BACI_HS92_V202601',
    project_root.parent.parent / 'ARG_Dashboard' / 'data' / 'input' / 'BACI_HS92_V202601',
]
baci_dir = next((candidate for candidate in baci_dir_candidates if candidate.exists()), baci_dir_candidates[0])
print(f'BACI source: {baci_dir}')
baci_files = [baci_dir / f'BACI_HS92_Y{year}_{BACI_VERSION}.csv' for year in YEARS]
missing_baci = [str(p) for p in baci_files if not p.exists()]
if missing_baci:
    raise FileNotFoundError('Missing BACI files: ' + '; '.join(missing_baci))

country_codes = pd.read_csv(baci_dir / f'country_codes_{BACI_VERSION}.csv', usecols=['country_code', 'country_iso3'])
country_codes['country_code'] = pd.to_numeric(country_codes['country_code'], errors='coerce').astype('Int64')
country_codes['country_iso3'] = country_codes['country_iso3'].astype(str).str.upper().str.strip().str[:3]
code_to_iso3 = dict(zip(country_codes['country_code'].dropna().astype(int), country_codes['country_iso3']))

parts = []
for f in baci_files:
    print(f'processing {f.name}', flush=True)
    for chunk in pd.read_csv(f, usecols=['t', 'i', 'j', 'k', 'v'], chunksize=1_000_000, low_memory=False):
        chunk['year'] = pd.to_numeric(chunk['t'], errors='coerce').astype('Int64')
        chunk = chunk[chunk['year'].isin(YEARS)].copy()
        if chunk.empty:
            continue
        chunk['iso3_o'] = pd.to_numeric(chunk['i'], errors='coerce').astype('Int64').map(code_to_iso3)
        chunk['iso3_d'] = pd.to_numeric(chunk['j'], errors='coerce').astype('Int64').map(code_to_iso3)
        chunk['hs92'] = (
            chunk['k'].astype(str)
            .str.replace(r'\.0$', '', regex=True)
            .str.replace(r'\D', '', regex=True)
            .str.zfill(6)
            .str[:4]
        )
        chunk['export_value'] = pd.to_numeric(chunk['v'], errors='coerce').fillna(0.0) * 1000.0
        chunk = chunk[
            chunk['iso3_o'].isin(valid_countries)
            & chunk['iso3_d'].isin(valid_countries)
            & chunk['hs92'].str.match(r'^\d{4}$', na=False)
            & ~chunk['hs92'].isin(['9999', 'XXXX'])
        ]
        if chunk.empty:
            continue
        parts.append(chunk.groupby(['year', 'iso3_o', 'iso3_d', 'hs92'], as_index=False)['export_value'].sum())

trade_hs4 = pd.concat(parts, ignore_index=True)
trade_hs4 = trade_hs4.groupby(['year', 'iso3_o', 'iso3_d', 'hs92'], as_index=False)['export_value'].sum()
trade_hs4.to_csv(INTERMEDIATE / 'trade_hs4_baci_2020_2024.csv', index=False)


print(f'trade_hs4 rows: {len(trade_hs4):,}')
trade_hs4.head()

BACI source: /Users/joc0445/Library/CloudStorage/OneDrive-HarvardUniversity/VS/Cordoba/ARG_Dashboard_V2/data/input/BACI_HS92_V202601
processing BACI_HS92_Y2020_V202601.csv


processing BACI_HS92_Y2021_V202601.csv


processing BACI_HS92_Y2022_V202601.csv


processing BACI_HS92_Y2023_V202601.csv


processing BACI_HS92_Y2024_V202601.csv


trade_hs4 rows: 19,682,206


,year,iso3_o,iso3_d,hs92,export_value
0,2020,AFG,AGO,3004,198.0
1,2020,AFG,AGO,8409,1287.0
2,2020,AFG,ARE,0106,16.0
3,2020,AFG,ARE,0202,284.0
4,2020,AFG,ARE,0203,162.0


## 3. Product Growth, Effective Exporters, and Travelled Distance

In [4]:
product = trade_hs4.groupby(['year', 'hs92'], as_index=False)['export_value'].sum()
product_2020_2024 = product[product['year'].isin([2020, 2024])].pivot(index='hs92', columns='year', values='export_value').reset_index()
for year in [2020, 2024]:
    if year not in product_2020_2024.columns:
        product_2020_2024[year] = 0.0
product_2020_2024['5yr_growth'] = np.where(
    (product_2020_2024[2020] > 0) & (product_2020_2024[2024] > 0),
    (product_2020_2024[2024] / product_2020_2024[2020]) ** (1 / 5) - 1,
    0.0,
)
product_2020_2024['above_median'] = product_2020_2024['5yr_growth'] > product_2020_2024['5yr_growth'].median()
product_growth = product_2020_2024[['hs92', '5yr_growth', 'above_median']].copy()
product_growth.to_csv(INTERMEDIATE / 'hs92_growth.csv', index=False)

# Average bilateral HS4 trade over 2020-2024, same as Ecuador reference.
trade_avg = trade_hs4.groupby(['iso3_o', 'iso3_d', 'hs92'], as_index=False)['export_value'].mean()

distances = pd.read_csv(INTERMEDIATE / 'bilateral_distances.csv')
distances['iso3_o'] = distances['iso3_o'].astype(str).str.upper().str.strip()
distances['iso3_d'] = distances['iso3_d'].astype(str).str.upper().str.strip()
distances['dist'] = pd.to_numeric(distances['dist'], errors='coerce')

trade_avg = trade_avg[trade_avg['iso3_o'].isin(valid_countries) & trade_avg['iso3_d'].isin(valid_countries)].copy()
trade_avg = trade_avg.merge(distances, on=['iso3_o', 'iso3_d'], how='left')
trade_avg['export_sq'] = trade_avg['export_value'] ** 2
trade_avg['dist_x_val'] = trade_avg['dist'] * trade_avg['export_value']

grouped = trade_avg.groupby('hs92').agg(
    total_export=('export_value', 'sum'),
    sum_sq_export=('export_sq', 'sum'),
    sum_weighted_dist=('dist_x_val', 'sum'),
)
grouped['eff_num_exp'] = np.where(grouped['sum_sq_export'] > 0, (grouped['total_export'] ** 2) / grouped['sum_sq_export'], 0.0)
grouped['travelled_distance'] = np.where(grouped['total_export'] > 0, grouped['sum_weighted_dist'] / grouped['total_export'], 0.0)
result_df = grouped[['eff_num_exp', 'travelled_distance']].reset_index()
result_df = result_df.merge(product_growth, on='hs92', how='left')

hs92_ref = pd.read_csv(INPUT / 'hs92_4digits.csv', encoding='utf-8-sig')
hs92_ref['hs92'] = hs92_ref['product_hs92_code'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(4).str[:4]
hs92_ref = hs92_ref[~hs92_ref['hs92'].isin(['9999', 'XXXX'])][['hs92', 'product_name_short', 'sector', 'green_product']].drop_duplicates('hs92')
valid_hs92 = sorted(hs92_ref['hs92'].unique())
result_df = result_df.merge(hs92_ref, on='hs92', how='left')
result_df.to_csv(INTERMEDIATE / 'hs92_attributes.csv', index=False)
print(f'hs92 attributes rows: {len(result_df):,}')
result_df.head()

hs92 attributes rows: 1,218


,hs92,eff_num_exp,travelled_distance,5yr_growth,above_median,product_name_short,sector,green_product
0,0101,18.296906,3487.743079,0.009504,False,Horses,Agriculture,False
1,0102,25.886270,3185.416621,0.042343,True,Bovine,Agriculture,False
2,0103,16.553074,929.485351,0.027561,False,Swine,Agriculture,False
3,0104,19.331279,2129.797662,0.049642,True,Sheep,Agriculture,False
4,0105,31.207077,1791.983851,0.049646,True,Fowl,Agriculture,False


## 4. Accessible Market

A destination is accessible for origin `z` and product `i` if either:

1. bilateral distance from `z` to destination `y` is less than or equal to `travelled_distance_i`, or
2. `z` already exports at least USD 100M of product `i` to destination `y`.

This replaces the older potential-market-only distance rule.

In [5]:
trade_bilateral = trade_hs4.copy()
for col in ['iso3_o', 'iso3_d', 'hs92']:
    trade_bilateral[col] = trade_bilateral[col].astype(str).str.upper().str.strip()
trade_bilateral['hs92'] = trade_bilateral['hs92'].str.zfill(4)
trade_bilateral['export_value'] = pd.to_numeric(trade_bilateral['export_value'], errors='coerce').fillna(0.0)
trade_bilateral = trade_bilateral[trade_bilateral['iso3_o'].isin(valid_countries) & trade_bilateral['iso3_d'].isin(valid_countries)].copy()

travel_df = result_df[['hs92', 'travelled_distance']].copy()
travel_df['hs92'] = travel_df['hs92'].astype(str).str.zfill(4)
travel_df['travelled_distance'] = pd.to_numeric(travel_df['travelled_distance'], errors='coerce')

distance_df = distances[['iso3_o', 'iso3_d', 'dist']].copy()

total_imports = (
    trade_bilateral.groupby(['year', 'iso3_d', 'hs92'], as_index=False)['export_value']
    .sum()
    .rename(columns={'export_value': 'total_imports'})
)
import_base = total_imports.merge(travel_df, on='hs92', how='left')
actual_exports = trade_bilateral[['year', 'iso3_o', 'iso3_d', 'hs92', 'export_value']].copy()

origin_year_parts = []
arg_detail_parts = []
for origin in sorted(valid_countries):
    dist_o = distance_df[distance_df['iso3_o'].eq(origin)][['iso3_d', 'dist']].copy()
    if dist_o.empty:
        continue
    frame = import_base.merge(dist_o, on='iso3_d', how='left')
    frame['iso3_o'] = origin
    exp_o = actual_exports[actual_exports['iso3_o'].eq(origin)][['year', 'iso3_d', 'hs92', 'export_value']].copy()
    frame = frame.merge(exp_o, on=['year', 'iso3_d', 'hs92'], how='left')
    frame['export_value'] = pd.to_numeric(frame['export_value'], errors='coerce').fillna(0.0)
    frame['total_imports'] = pd.to_numeric(frame['total_imports'], errors='coerce').fillna(0.0)
    frame['travelled_distance'] = pd.to_numeric(frame['travelled_distance'], errors='coerce')
    frame['dist'] = pd.to_numeric(frame['dist'], errors='coerce')
    within_distance = frame['dist'].notna() & frame['travelled_distance'].notna() & (frame['dist'] <= frame['travelled_distance'])
    large_existing_flow = frame['export_value'] >= ACCESSIBLE_EXPORT_THRESHOLD_USD
    frame['accessible_market'] = (within_distance | large_existing_flow).astype(int)
    frame['accessible_market_imports'] = np.where(frame['accessible_market'].eq(1), frame['total_imports'], 0.0)
    origin_year_parts.append(
        frame.groupby(['iso3_o', 'hs92', 'year'], as_index=False)['accessible_market_imports']
        .sum()
        .rename(columns={'accessible_market_imports': 'accessible_market_size'})
    )
    if origin == FOCUS_ISO:
        arg_detail_parts.append(
            frame[['year', 'iso3_o', 'iso3_d', 'hs92', 'travelled_distance', 'dist', 'export_value', 'accessible_market', 'total_imports', 'accessible_market_imports']].copy()
        )

accessible_market_by_product_year = pd.concat(origin_year_parts, ignore_index=True)
accessible_markets = pd.concat(arg_detail_parts, ignore_index=True) if arg_detail_parts else pd.DataFrame()
accessible_markets.to_csv(INTERMEDIATE / 'accessible_market_by_country_product_year.csv', index=False)
accessible_markets_2024 = accessible_markets[accessible_markets['year'].eq(2024)].copy()
accessible_markets_2024.to_csv(INTERMEDIATE / 'accessible_market_by_country_product.csv', index=False)


accessible_market_by_product_year = accessible_market_by_product_year.groupby(['iso3_o', 'hs92', 'year'], as_index=False)['accessible_market_size'].sum()
accessible_market_by_product_year.to_csv(INTERMEDIATE / 'accessible_market_by_product_year.csv', index=False)
accessible_market_by_product_2024 = accessible_market_by_product_year[accessible_market_by_product_year['year'].eq(2024)].copy()
accessible_market_by_product_2024[['iso3_o', 'hs92', 'accessible_market_size']].to_csv(INTERMEDIATE / 'accessible_market_by_product.csv', index=False)

am_pivot = accessible_market_by_product_year.pivot_table(index=['iso3_o', 'hs92'], columns='year', values='accessible_market_size', aggfunc='sum', fill_value=0).reset_index()
for year in [2020, 2024]:
    if year not in am_pivot.columns:
        am_pivot[year] = 0.0
am_pivot['accessible_market_growth_5y'] = np.where(
    (am_pivot[2020] > 0) & (am_pivot[2024] > 0),
    (am_pivot[2024] / am_pivot[2020]) ** (1 / 5) - 1,
    0.0,
)
accessible_market_growth = am_pivot[['iso3_o', 'hs92', 2020, 2024, 'accessible_market_growth_5y']].rename(columns={
    2020: 'accessible_market_size_2020',
    2024: 'accessible_market_size_2024',
})
accessible_market_growth.to_csv(INTERMEDIATE / 'accessible_market_growth_by_product.csv', index=False)


print(f'accessible market country-product-year rows: {len(accessible_markets):,}')
print(f'accessible market aggregate rows: {len(accessible_market_by_product_year):,}')
accessible_market_growth[accessible_market_growth['iso3_o'].eq(FOCUS_ISO)].head()

accessible market country-product-year rows: 828,247
accessible market aggregate rows: 882,180


year,iso3_o,hs92,accessible_market_size_2020,accessible_market_size_2024,accessible_market_growth_5y
4872,ARG,0101,9340827.0,20520831.0,0.170474
4873,ARG,0102,2463284.0,2410793.0,-0.004299
4874,ARG,0103,70307.0,164128.0,0.184778
4875,ARG,0104,172106.0,121882.0,-0.066684
4876,ARG,0105,21638875.0,26123495.0,0.038387


## 5. Complexity and Proximity with `ecomplexity`

Manual RCA is computed from BACI exports and transformed as `rca / (rca + 1)`. This transformed RCA is passed to `ecomplexity(..., presence_test='manual')`, matching the Ecuador reference.

In [6]:
trade = (
    trade_hs4.groupby(['year', 'iso3_o', 'hs92'], as_index=False)['export_value']
    .sum()
    .rename(columns={'year': 'time', 'iso3_o': 'location', 'hs92': 'product', 'export_value': 'value'})
)
trade['product'] = trade['product'].astype(str).str.zfill(4)
trade['location'] = trade['location'].astype(str).str.upper().str.strip()
trade = trade[trade['product'].isin(valid_hs92) & trade['location'].isin(valid_countries)].copy()

full_panel = pd.MultiIndex.from_product([YEARS, sorted(valid_countries), valid_hs92], names=['time', 'location', 'product']).to_frame(index=False)
trade = full_panel.merge(trade, on=['time', 'location', 'product'], how='left')
trade['value'] = pd.to_numeric(trade['value'], errors='coerce').fillna(0.0)

country_total = trade.groupby(['time', 'location'])['value'].transform('sum')
product_total = trade.groupby(['time', 'product'])['value'].transform('sum')
world_total = trade.groupby('time')['value'].transform('sum')
trade['rca'] = np.where(
    (country_total > 0) & (product_total > 0) & (world_total > 0),
    (trade['value'] / country_total) / (product_total / world_total),
    0.0,
)
trade['rca'] = pd.to_numeric(trade['rca'], errors='coerce').replace([np.inf, -np.inf], 0.0).fillna(0.0)
trade['rca_transformation'] = trade['rca'] / (trade['rca'] + 1.0)

trade_ready = trade[['time', 'location', 'product', 'rca_transformation']].copy()
trade_cols = {'time': 'time', 'loc': 'location', 'prod': 'product', 'val': 'rca_transformation'}

complexity_all = ecomplexity(trade_ready, trade_cols, presence_test='manual')
complexity_all = complexity_all.drop(columns=['rca'], errors='ignore').merge(
    trade[['time', 'location', 'product', 'rca', 'rca_transformation']],
    on=['time', 'location', 'product', 'rca_transformation'],
    how='left',
)
if 'density' in complexity_all.columns:
    density_rank = complexity_all.groupby(['time', 'product'])['density'].rank(method='average', ascending=True)
    density_n = complexity_all.groupby(['time', 'product'])['density'].transform('count')
    complexity_all['density_percentile'] = np.where(density_n > 1, (density_rank - 1.0) / (density_n - 1.0), 0.0)
else:
    complexity_all['density_percentile'] = 0.0

complexity_numeric_cols = ['rca_transformation', 'diversity', 'ubiquity', 'mcp', 'eci', 'pci', 'density', 'coi', 'cog', 'rca', 'density_percentile']
for col in complexity_numeric_cols:
    if col in complexity_all.columns:
        complexity_all[col] = pd.to_numeric(complexity_all[col], errors='coerce').replace([np.inf, -np.inf], 0.0).fillna(0.0)

complexity_all.to_csv(INTERMEDIATE / 'complexity_calculations.csv', index=False)
complexity_arg_2024 = complexity_all[(complexity_all['location'].eq(FOCUS_ISO)) & (complexity_all['time'].eq(2024))].copy()
complexity_arg_2024.to_csv(INTERMEDIATE / 'complexity_arg_2024.csv', index=False)

prox_df = proximity(trade_ready, trade_cols, presence_test='manual')
prox_df['product_1'] = prox_df['product_1'].astype(str).str.zfill(4)
prox_df['product_2'] = prox_df['product_2'].astype(str).str.zfill(4)
prox_df['proximity'] = pd.to_numeric(prox_df['proximity'], errors='coerce').replace([np.inf, -np.inf], 0.0).fillna(0.0)
prox_df.to_csv(INTERMEDIATE / 'proximity.csv', index=False)

print(f'complexity rows: {len(complexity_all):,}')
print(f'ARG 2024 complexity rows: {len(complexity_arg_2024):,}')
print(f'proximity rows: {len(prox_df):,}')
complexity_arg_2024.head()

2020


Percentage of pairs compared that meet log-supermodularity condition: 43.24%
2021


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ecomplexity/ecomplexity.py:252: UserWarning: Year 2020: Log-supermodularity condition is not fully satisfied (43.24% of pairs compared satisfy this condition). The ECI and PCI values may not be a true representation of the complexity. More details at: https://growthlab.hks.harvard.edu/publications/structural-ranking-economic-complexity
  warnings.warn(


Percentage of pairs compared that meet log-supermodularity condition: 38.63%
2022


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ecomplexity/ecomplexity.py:252: UserWarning: Year 2021: Log-supermodularity condition is not fully satisfied (38.63% of pairs compared satisfy this condition). The ECI and PCI values may not be a true representation of the complexity. More details at: https://growthlab.hks.harvard.edu/publications/structural-ranking-economic-complexity
  warnings.warn(


...

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ecomplexity/ecomplexity.py:252: UserWarning: Year 2024: Log-supermodularity condition is not fully satisfied (39.17% of pairs compared satisfy this condition). The ECI and PCI values may not be a true representation of the complexity. More details at: https://growthlab.hks.harvard.edu/publications/structural-ranking-economic-complexity
  warnings.warn(


2020
2021
2022
2023


2024


complexity rows: 899,725
ARG 2024 complexity rows: 1,241
proximity rows: 7,403,012


,location,product,rca_transformation,time,diversity,ubiquity,mcp,eci,pci,density,coi,cog,rca,density_percentile
724744,ARG,0101,0.684054,2024,205.71297,17.658278,0.684054,0.160719,1.352446,0.177375,0.666354,0.145698,2.165096,0.615385
724745,ARG,0102,0.004196,2024,205.71297,35.281952,0.004196,0.160719,-0.384204,0.196264,0.666354,0.193634,0.004214,0.566434
724746,ARG,0103,0.003001,2024,205.71297,14.452012,0.003001,0.160719,2.816817,0.161392,0.666354,0.557105,0.003011,0.573427
724747,ARG,0104,0.004388,2024,205.71297,27.502449,0.004388,0.160719,-1.924155,0.195080,0.666354,0.053263,0.004407,0.566434
724748,ARG,0105,0.126681,2024,205.71297,32.435746,0.126681,0.160719,0.091132,0.185853,0.666354,0.234823,0.145057,0.552448


## 6. Demand Alignment Index (DAI) and Opportunity Metrics

DAI is computed directly from BACI partner demand and replaces the previous network-alignment naming in both source files and dashboard labels.

In [7]:
bilateral_2024 = trade_hs4[trade_hs4['year'].eq(2024)].copy()

xzy = bilateral_2024.groupby(['iso3_o', 'iso3_d'], as_index=False)['export_value'].sum().rename(columns={'iso3_o': 'exporter_iso', 'iso3_d': 'importer', 'export_value': 'X_zy'})
xzi = bilateral_2024.groupby(['iso3_o', 'hs92'], as_index=False)['export_value'].sum().rename(columns={'iso3_o': 'exporter_iso', 'hs92': 'product_code', 'export_value': 'X_zi'})
miy = bilateral_2024.groupby(['iso3_d', 'hs92'], as_index=False)['export_value'].sum().rename(columns={'iso3_d': 'importer', 'hs92': 'product_code', 'export_value': 'M_iy'})

M_y = xzy.groupby('importer')['X_zy'].sum().rename('M_y')
X_z = xzy.groupby('exporter_iso')['X_zy'].sum().rename('X_z')
WT = float(M_y.sum())
pipe = xzy.merge(M_y.reset_index(), on='importer', how='left').merge(X_z.reset_index(), on='exporter_iso', how='left')
pipe['export_share_to_y'] = np.where(pipe['X_z'] > 0, pipe['X_zy'] / pipe['X_z'], 0.0)
pipe['market_share_world_trade'] = np.where(WT > 0, pipe['M_y'] / WT, 0.0)
pipe['dai_affinity'] = np.where(pipe['market_share_world_trade'] > 0, pipe['export_share_to_y'] / pipe['market_share_world_trade'], 0.0)
pipe['dai_affinity'] = pipe['dai_affinity'].replace([np.inf, -np.inf], 0.0).fillna(0.0)

world_import_i = miy.groupby('product_code')['M_iy'].sum().rename('world_import_i')
demand = miy.merge(world_import_i.reset_index(), on='product_code', how='left')
demand['omega'] = np.where(demand['world_import_i'] > 0, demand['M_iy'] / demand['world_import_i'], 0.0)
demand['omega'] = demand['omega'].replace([np.inf, -np.inf], 0.0).fillna(0.0)

partners = sorted(set(pipe['importer']).union(set(demand['importer'])))
exporters = sorted(valid_countries)
products = valid_hs92
affinity_mat = pipe.pivot(index='exporter_iso', columns='importer', values='dai_affinity').reindex(index=exporters, columns=partners, fill_value=0.0).fillna(0.0)
omega_mat = demand.pivot(index='importer', columns='product_code', values='omega').reindex(index=partners, columns=products, fill_value=0.0).fillna(0.0)
dai_arr = affinity_mat.to_numpy(dtype=float) @ omega_mat.to_numpy(dtype=float)

dai = (
    pd.DataFrame(dai_arr, index=exporters, columns=products)
    .stack()
    .rename('dai_index')
    .reset_index()
    .rename(columns={'level_0': 'exporter_iso', 'level_1': 'hs4'})
)
top30 = xzi.sort_values(['product_code', 'X_zi', 'exporter_iso'], ascending=[True, False, True]).groupby('product_code', sort=False).head(30)[['product_code', 'exporter_iso']].rename(columns={'product_code': 'hs4'})
focus_rows = pd.DataFrame({'hs4': products, 'exporter_iso': FOCUS_ISO})
comparison_set = pd.concat([top30, focus_rows], ignore_index=True).drop_duplicates()
dai = dai.merge(comparison_set, on=['hs4', 'exporter_iso'], how='inner')
dai['dai_percentile'] = dai.groupby('hs4')['dai_index'].rank(method='average', pct=True) * 100

top5 = xzi[xzi['exporter_iso'].ne(FOCUS_ISO)].sort_values(['product_code', 'X_zi', 'exporter_iso'], ascending=[True, False, True]).groupby('product_code', sort=False).head(5)[['product_code', 'exporter_iso']].rename(columns={'product_code': 'hs4'})
comp = top5.merge(dai[['hs4', 'exporter_iso', 'dai_percentile']], on=['hs4', 'exporter_iso'], how='left')
comp_med = comp.groupby('hs4', as_index=False).agg(competitor_median_dai=('dai_percentile', 'median'))
arg_dai = dai[dai['exporter_iso'].eq(FOCUS_ISO)][['hs4', 'dai_index', 'dai_percentile']].copy()
arg_dai = arg_dai.merge(comp_med, on='hs4', how='left')
arg_dai['dai_lead'] = arg_dai['dai_percentile'].fillna(0.0) - arg_dai['competitor_median_dai'].fillna(0.0)
arg_dai = arg_dai[['hs4', 'dai_index', 'dai_percentile', 'dai_lead']]
arg_dai.head()

,hs4,dai_index,dai_percentile,dai_lead
0,0101,0.554408,10.000000,-66.666667
1,0102,0.738540,12.903226,-61.290323
2,0103,0.443042,6.451613,-41.935484
3,0104,1.342093,64.516129,-12.903226
4,0105,1.047235,29.032258,-22.580645


In [8]:
world = trade_hs4.groupby(['year', 'hs92'], as_index=False)['export_value'].sum().rename(columns={'export_value': 'world_value'})
world_pivot = world.pivot(index='hs92', columns='year', values='world_value').reindex(valid_hs92).fillna(0.0).reset_index().rename(columns={'hs92': 'hs4'})
for year in [2020, 2024]:
    if year not in world_pivot.columns:
        world_pivot[year] = 0.0
world_pivot['market_growth_5y'] = np.where((world_pivot[2020] > 0) & (world_pivot[2024] > 0), (world_pivot[2024] / world_pivot[2020]) ** (1 / 5) - 1, 0.0)
world_2024 = world[world['year'].eq(2024)][['hs92', 'world_value']].rename(columns={'hs92': 'hs4', 'world_value': 'total_trade'})
world_total_2024 = float(world_2024['total_trade'].sum())
world_2024['market_size_share'] = np.where(world_total_2024 > 0, world_2024['total_trade'] / world_total_2024, 0.0)

country_year = trade_hs4[trade_hs4['iso3_o'].eq(FOCUS_ISO)].groupby(['year', 'hs92'], as_index=False)['export_value'].sum().rename(columns={'hs92': 'hs4', 'export_value': 'country_value'})
country_pivot = country_year.pivot(index='hs4', columns='year', values='country_value').reindex(valid_hs92).fillna(0.0).reset_index()
for year in [2020, 2024]:
    if year not in country_pivot.columns:
        country_pivot[year] = 0.0
country_pivot['country_export_growth_5y'] = np.where((country_pivot[2020] > 0) & (country_pivot[2024] > 0), (country_pivot[2024] / country_pivot[2020]) ** (1 / 5) - 1, 0.0)
country_2024 = country_pivot[['hs4', 2024, 'country_export_growth_5y']].rename(columns={2024: 'country_current_exports'})

share = world_pivot[['hs4', 2020, 2024]].rename(columns={2020: 'world_2020', 2024: 'world_2024'})
share = share.merge(country_pivot[['hs4', 2020, 2024]].rename(columns={2020: 'country_2020', 2024: 'country_2024'}), on='hs4', how='left').fillna(0.0)
share['country_market_share_2020'] = np.where(share['world_2020'] > 0, share['country_2020'] / share['world_2020'], 0.0)
share['country_market_share_2024'] = np.where(share['world_2024'] > 0, share['country_2024'] / share['world_2024'], 0.0)
share['market_share_change_abs'] = share['country_market_share_2024'] - share['country_market_share_2020']

attrs = result_df.rename(columns={'hs92': 'hs4'})[['hs4', 'eff_num_exp', 'travelled_distance']].rename(columns={'travelled_distance': 'distance_travelled'})

exp_hs4 = bilateral_2024.groupby(['iso3_o', 'hs92'], as_index=False)['export_value'].sum().rename(columns={'iso3_o': 'exporter', 'hs92': 'hs4', 'export_value': 'value'})
totals = exp_hs4.groupby('hs4')['value'].sum().rename('total')
exp_hs4 = exp_hs4.merge(totals, on='hs4', how='left')
exp_hs4['share'] = np.where(exp_hs4['total'] > 0, exp_hs4['value'] / exp_hs4['total'], 0.0)
eff = exp_hs4.groupby('hs4', as_index=False)['share'].apply(lambda s: float(1 / np.sum(np.square(s))) if np.sum(np.square(s)) > 0 else 0.0).rename(columns={'share': 'eff_num_exp'})
exp_hs4['country_exporter_rank'] = exp_hs4.groupby('hs4')['value'].rank(method='min', ascending=False)
rank = exp_hs4[exp_hs4['exporter'].eq(FOCUS_ISO)][['hs4', 'country_exporter_rank']]

am = accessible_market_growth[accessible_market_growth['iso3_o'].eq(FOCUS_ISO)].rename(columns={
    'hs92': 'hs4',
    'accessible_market_size_2024': 'accessible_market_size',
})[['hs4', 'accessible_market_size', 'accessible_market_growth_5y']]

out = world_pivot[['hs4', 'market_growth_5y']].merge(world_2024, on='hs4', how='outer')
out = out.merge(country_2024, on='hs4', how='left')
out = out.merge(share[['hs4', 'country_market_share_2020', 'country_market_share_2024', 'market_share_change_abs']], on='hs4', how='left')
out = out.merge(attrs, on='hs4', how='left')
out = out.drop(columns=['eff_num_exp'], errors='ignore').merge(eff, on='hs4', how='left')
out = out.merge(rank, on='hs4', how='left')
out = out.merge(am, on='hs4', how='left')
out = out.merge(arg_dai, on='hs4', how='left')
out = out.fillna(0.0)

country_total_2024 = float(out['country_current_exports'].sum())
out['raw_rca_trade'] = np.where((country_total_2024 > 0) & (world_total_2024 > 0) & (out['total_trade'] > 0), (out['country_current_exports'] / country_total_2024) / (out['total_trade'] / world_total_2024), 0.0)
out['market_size'] = out['total_trade']
total_accessible = float(out['accessible_market_size'].sum())
out['accessible_market_size_share'] = np.where(total_accessible > 0, out['accessible_market_size'] / total_accessible, 0.0)
out['accessible_market_to_market_ratio'] = np.where(out['market_size'] > 0, out['accessible_market_size'] / out['market_size'], 0.0)


out['above_median_cagr'] = out['market_growth_5y'] > out['market_growth_5y'].median()
out['above_median_export_cagr'] = out['country_export_growth_5y'] > out['country_export_growth_5y'].median()
out['above_median_accessible_market_growth'] = out['accessible_market_growth_5y'] > out['accessible_market_growth_5y'].median()

out.to_csv(INTERMEDIATE / 'opportunity_metrics_hs4_arg.csv', index=False)
print(f'opportunity metrics rows: {len(out):,}')
out.head()

opportunity metrics rows: 1,241


,hs4,market_growth_5y,total_trade,market_size_share,country_current_exports,country_export_growth_5y,country_market_share_2020,country_market_share_2024,market_share_change_abs,distance_travelled,...,dai_index,dai_percentile,dai_lead,raw_rca_trade,market_size,accessible_market_size_share,accessible_market_to_market_ratio,above_median_cagr,above_median_export_cagr,above_median_accessible_market_growth
0,0101,0.009504,4.059211e+09,0.000188,30883241.0,0.129162,0.004345,0.007608,0.003263,3487.743079,...,0.554408,10.000000,-66.666667,2.165096,4.059211e+09,1.439620e-05,0.005055,False,True,True
1,0102,0.042343,1.127415e+10,0.000522,166937.0,-0.169256,0.000046,0.000015,-0.000031,3185.416621,...,0.738540,12.903226,-61.290323,0.004214,1.127415e+10,1.691269e-06,0.000214,True,False,False
2,0103,0.027561,4.902724e+09,0.000227,51866.0,-0.059028,0.000016,0.000011,-0.000006,929.485351,...,0.443042,6.451613,-41.935484,0.003011,4.902724e+09,1.151425e-07,0.000033,False,False,True
3,0104,0.049642,2.044997e+09,0.000095,31670.0,0.123262,0.000011,0.000015,0.000004,2129.797662,...,1.342093,64.516129,-12.903226,0.004407,2.044997e+09,8.550518e-08,0.000060,True,True,False
4,0105,0.049646,3.883969e+09,0.000180,1979783.0,0.025628,0.000572,0.000510,-0.000063,1791.983851,...,1.047235,29.032258,-22.580645,0.145057,3.883969e+09,1.832669e-05,0.006726,True,True,False


## 7. Anchored Proximity Outputs

This section ports the Ecuador anchored-proximity logic to Argentina. It uses BACI-derived exports, average 2020-2024 complexity/proximity from `ecomplexity`, persistent ARG RCA anchors, and the BACI/DAI/accessible-market metrics produced above.


In [9]:
ANCHOR_RCA_THRESHOLD = 1.0
ANCHOR_MIN_YEARS = 3
PERIOD_LABEL = '2020_2024_avg'

HS_SECTION_RULES = [
    (1, range(1, 6), '1. Live animals; animal products'),
    (2, range(6, 15), '2. Vegetable products'),
    (3, range(15, 16), '3. Animal or vegetable fats and oils'),
    (4, range(16, 25), '4. Prepared foodstuffs; beverages, spirits and tobacco'),
    (5, range(25, 28), '5. Mineral products'),
    (6, range(28, 39), '6. Products of the chemical or allied industries'),
    (7, range(39, 41), '7. Plastics and articles thereof; rubber and articles thereof'),
    (8, range(41, 44), '8. Raw hides and skins, leather, furskins and articles thereof'),
    (9, range(44, 47), '9. Wood and articles of wood'),
    (10, range(47, 50), '10. Pulp, paper and paperboard'),
    (11, range(50, 64), '11. Textiles and textile articles'),
    (12, range(64, 68), '12. Footwear, headgear, umbrellas and related articles'),
    (13, range(68, 71), '13. Articles of stone, plaster, cement, ceramics and glass'),
    (14, range(71, 72), '14. Pearls, precious stones and metals'),
    (15, range(72, 84), '15. Base metals and articles of base metal'),
    (16, range(84, 86), '16. Machinery, mechanical appliances and electrical equipment'),
    (17, range(86, 90), '17. Vehicles, aircraft, vessels and associated transport equipment'),
    (18, range(90, 93), '18. Optical, photographic, medical and musical instruments'),
    (19, range(93, 94), '19. Arms and ammunition'),
    (20, range(94, 97), '20. Miscellaneous manufactured articles'),
    (21, range(97, 98), "21. Works of art, collectors' pieces and antiques"),
]

def hs4_to_section_name(code: str) -> str:
    digits = ''.join(ch for ch in str(code) if ch.isdigit())
    if len(digits) < 2:
        return 'Other'
    chapter = int(digits[:2])
    for _, chapters, label in HS_SECTION_RULES:
        if chapter in chapters:
            return label
    return 'Other'

def minmax(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    lo = s.min(skipna=True)
    hi = s.max(skipna=True)
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.0, index=series.index)
    return ((s - lo) / (hi - lo)).fillna(0.0)

def add_density_percentile_avg(df: pd.DataFrame) -> pd.DataFrame:
    out_df = df.copy()
    if 'density' in out_df.columns:
        density_rank = out_df.groupby(['time', 'product'])['density'].rank(method='average', ascending=True)
        density_n = out_df.groupby(['time', 'product'])['density'].transform('count')
        out_df['density_percentile'] = np.where(density_n > 1, (density_rank - 1.0) / (density_n - 1.0), 0.0)
    else:
        out_df['density_percentile'] = 0.0
    return out_df

def add_embeddedness_avg(complexity_df: pd.DataFrame, proximity_df: pd.DataFrame) -> pd.DataFrame:
    products = sorted(complexity_df['product'].astype(str).str.zfill(4).unique())
    countries = sorted(complexity_df['location'].astype(str).str.upper().unique())
    phi = proximity_df.pivot(index='product_1', columns='product_2', values='proximity').reindex(index=products, columns=products).fillna(0.0)
    phi_values = phi.to_numpy(dtype=float)
    np.fill_diagonal(phi_values, 0.0)
    mcp = complexity_df.pivot(index='location', columns='product', values='mcp').reindex(index=countries, columns=products).fillna(0.0)
    embedded = mcp.to_numpy(dtype=float).dot(phi_values)
    embedded_df = pd.DataFrame(embedded, index=mcp.index, columns=mcp.columns).stack().rename('embeddedness').reset_index()
    embedded_df.columns = ['location', 'product', 'embeddedness']
    return complexity_df.merge(embedded_df, on=['location', 'product'], how='left')

# Average 2020-2024 country-product exports, using the BACI-derived full panel built above.
trade_avg = trade.groupby(['location', 'product'], as_index=False)['value'].mean()
trade_avg.insert(0, 'time', PERIOD_LABEL)

country_total_avg = trade_avg.groupby(['time', 'location'])['value'].transform('sum')
product_total_avg = trade_avg.groupby(['time', 'product'])['value'].transform('sum')
world_total_avg = trade_avg.groupby('time')['value'].transform('sum')
trade_avg['rca'] = np.where(
    (country_total_avg > 0) & (product_total_avg > 0) & (world_total_avg > 0),
    (trade_avg['value'] / country_total_avg) / (product_total_avg / world_total_avg),
    0.0,
)
trade_avg['rca'] = pd.to_numeric(trade_avg['rca'], errors='coerce').replace([np.inf, -np.inf], 0.0).fillna(0.0)
trade_avg['rca_transformation'] = trade_avg['rca'] / (trade_avg['rca'] + 1.0)
trade_avg_ready = trade_avg[['time', 'location', 'product', 'rca_transformation']].copy()
avg_cols = {'time': 'time', 'loc': 'location', 'prod': 'product', 'val': 'rca_transformation'}

complexity_avg = ecomplexity(trade_avg_ready, avg_cols, presence_test='manual')
complexity_avg = complexity_avg.drop(columns=['rca'], errors='ignore').merge(
    trade_avg[['time', 'location', 'product', 'rca', 'rca_transformation']],
    on=['time', 'location', 'product', 'rca_transformation'],
    how='left',
)
complexity_avg = add_density_percentile_avg(complexity_avg)
prox_avg = proximity(trade_avg_ready, avg_cols, presence_test='manual')
prox_avg['product_1'] = prox_avg['product_1'].astype(str).str.zfill(4)
prox_avg['product_2'] = prox_avg['product_2'].astype(str).str.zfill(4)
prox_avg['proximity'] = pd.to_numeric(prox_avg['proximity'], errors='coerce').replace([np.inf, -np.inf], 0.0).fillna(0.0)
prox_avg.loc[prox_avg['product_1'].eq(prox_avg['product_2']), 'proximity'] = 0.0
complexity_avg['product'] = complexity_avg['product'].astype(str).str.zfill(4)
complexity_avg['location'] = complexity_avg['location'].astype(str).str.upper().str.strip()
complexity_avg = add_embeddedness_avg(complexity_avg, prox_avg)

for col in ['rca_transformation', 'diversity', 'ubiquity', 'mcp', 'eci', 'pci', 'density', 'coi', 'cog', 'rca', 'density_percentile', 'embeddedness']:
    if col in complexity_avg.columns:
        complexity_avg[col] = pd.to_numeric(complexity_avg[col], errors='coerce').replace([np.inf, -np.inf], 0.0).fillna(0.0)

complexity_avg.to_csv(INTERMEDIATE / 'complexity_calculations_avg.csv', index=False)
prox_avg.to_csv(INTERMEDIATE / 'proximity_avg.csv', index=False)

# Persistent ARG RCA anchors: RCA > 1 in at least 3 of 5 BACI years.
annual_arg = complexity_all[
    complexity_all['location'].eq(FOCUS_ISO)
    & complexity_all['time'].isin(YEARS)
    & complexity_all['product'].astype(str).str.match(r'^\d{4}$', na=False)
].copy()
annual_arg['product'] = annual_arg['product'].astype(str).str.zfill(4)
annual_arg['rca'] = pd.to_numeric(annual_arg['rca'], errors='coerce').replace([np.inf, -np.inf], 0.0).fillna(0.0)
presence = annual_arg.pivot_table(index='product', columns='time', values='rca', aggfunc='first').reindex(valid_hs92).fillna(0.0)
for year in YEARS:
    if year not in presence.columns:
        presence[year] = 0.0
presence = presence[YEARS].copy()
presence['rca_gt_1_years'] = (presence[YEARS] > ANCHOR_RCA_THRESHOLD).sum(axis=1)
presence['presence'] = (presence['rca_gt_1_years'] >= ANCHOR_MIN_YEARS).astype(int)
presence = presence.reset_index().rename(columns={'product': 'hs4', **{year: f'rca_{year}' for year in YEARS}})
presence.to_csv(INTERMEDIATE / f'{FOCUS_COUNTRY_SLUG}_presence.csv', index=False)

labels = pd.read_csv(INPUT / 'hs92_4digits.csv', dtype={'product_hs92_code': 'string'}, encoding='utf-8-sig')
labels['hs4'] = labels['product_hs92_code'].astype(str).str.zfill(4)
labels = labels[['hs4', 'product_name_short', 'sector']].drop_duplicates('hs4').copy()
arg_avg = complexity_avg[complexity_avg['location'].eq(FOCUS_ISO)][['product', 'density', 'embeddedness']].rename(columns={'product': 'hs4'})
anchors = presence[presence['presence'].eq(1)].merge(labels, on='hs4', how='left').merge(arg_avg, on='hs4', how='left')
anchors = anchors[
    ['hs4', 'product_name_short', 'sector', 'rca_2020', 'rca_2021', 'rca_2022', 'rca_2023', 'rca_2024', 'rca_gt_1_years', 'presence', 'density', 'embeddedness']
].sort_values('hs4')
anchors.to_csv(INTERMEDIATE / 'anchors.csv', index=False)

anchor_codes = set(anchors['hs4'].astype(str).str.zfill(4))
anchor_info = anchors[['hs4', 'product_name_short', 'sector']].rename(columns={'hs4': 'anchor_hs4', 'product_name_short': 'anchor_product_name_short', 'sector': 'anchor_sector'})
anchor_metrics = anchors[['hs4', 'density', 'embeddedness']].rename(columns={'hs4': 'anchor_hs4', 'density': 'anchor_density', 'embeddedness': 'anchor_embeddedness'})
density_rank = anchor_metrics['anchor_density'].rank(method='average', ascending=True)
density_n = anchor_metrics['anchor_density'].notna().sum()
anchor_metrics['anchor_density_percentile'] = np.where(density_n > 1, (density_rank - 1.0) / (density_n - 1.0) * 100.0, 0.0)

label_info = labels.rename(columns={'hs4': 'candidate_hs4', 'product_name_short': 'candidate_product_name_short', 'sector': 'candidate_sector'})
complexity_candidate = complexity_avg[['product', 'pci', 'cog']].rename(columns={'product': 'candidate_hs4'}).drop_duplicates('candidate_hs4')
opportunity_cols = [
    'hs4', 'distance_travelled', 'accessible_market_size', 'accessible_market_size_share',
    'accessible_market_growth_5y', 'dai_index', 'dai_percentile', 'dai_lead',
]
opportunity = out[opportunity_cols].copy()
opportunity['candidate_hs4'] = opportunity['hs4'].astype(str).str.zfill(4)
opportunity = opportunity.drop(columns=['hs4']).drop_duplicates('candidate_hs4')

global_proximity_median = float(prox_avg['proximity'].median()) if not prox_avg.empty else 0.0
panel = prox_avg[prox_avg['product_1'].isin(anchor_codes) & ~prox_avg['product_2'].isin(anchor_codes)].copy()
panel = panel.rename(columns={'product_1': 'anchor_hs4', 'product_2': 'candidate_hs4'})
panel = panel.sort_values(['anchor_hs4', 'proximity', 'candidate_hs4'], ascending=[True, False, True])
panel['proximity_rank'] = panel.groupby('anchor_hs4').cumcount() + 1
panel['eligible_candidate_count'] = panel.groupby('anchor_hs4')['candidate_hs4'].transform('count')
panel['top_1pct_cutoff_n'] = panel['eligible_candidate_count'].map(lambda n: int(math.ceil(n * 0.01)))
panel = panel[panel['proximity_rank'] <= panel['top_1pct_cutoff_n']].copy()

panel = (
    panel.merge(anchor_info, on='anchor_hs4', how='left')
    .merge(anchor_metrics, on='anchor_hs4', how='left')
    .merge(label_info, on='candidate_hs4', how='left')
    .merge(complexity_candidate, on='candidate_hs4', how='left')
    .merge(opportunity, on='candidate_hs4', how='left')
)
panel['proximity_above_country_median'] = (pd.to_numeric(panel['proximity'], errors='coerce').fillna(0.0) > global_proximity_median).astype(int)
panel['anchor_hs_section_name'] = panel['anchor_hs4'].map(hs4_to_section_name)
panel['candidate_hs_section_name'] = panel['candidate_hs4'].map(hs4_to_section_name)

candidate_score_base = panel[[
    'candidate_hs4', 'pci', 'cog', 'accessible_market_size', 'accessible_market_growth_5y', 'dai_index', 'dai_percentile'
]].drop_duplicates('candidate_hs4').copy()
for col in ['pci', 'cog', 'accessible_market_size', 'accessible_market_growth_5y', 'dai_index', 'dai_percentile']:
    candidate_score_base[col] = pd.to_numeric(candidate_score_base[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0.0)
candidate_score_base['pci_norm'] = minmax(candidate_score_base['pci'])
candidate_score_base['cog_norm'] = minmax(candidate_score_base['cog'])
candidate_score_base['accessible_market_size_norm'] = minmax(candidate_score_base['accessible_market_size'])
candidate_score_base['accessible_market_growth_norm'] = minmax(candidate_score_base['accessible_market_growth_5y'])
candidate_score_base['dai_norm'] = minmax(candidate_score_base['dai_percentile'])
candidate_score_base['attractiveness_score'] = (
    0.35 * candidate_score_base['pci_norm']
    + 0.35 * candidate_score_base['cog_norm']
    + 0.15 * candidate_score_base['accessible_market_size_norm']
    + 0.15 * candidate_score_base['accessible_market_growth_norm']
)
candidate_score_base['feasibility_score'] = candidate_score_base['dai_norm']
candidate_score_base['combined_score'] = 0.70 * candidate_score_base['attractiveness_score'] + 0.30 * candidate_score_base['feasibility_score']

panel = panel.merge(
    candidate_score_base[[
        'candidate_hs4', 'pci_norm', 'cog_norm', 'accessible_market_size_norm', 'accessible_market_growth_norm',
        'dai_norm', 'attractiveness_score', 'feasibility_score', 'combined_score',
    ]],
    on='candidate_hs4',
    how='left',
)
anchor_output_cols = [
    'anchor_hs4', 'anchor_product_name_short', 'anchor_sector', 'anchor_hs_section_name',
    'anchor_density', 'anchor_density_percentile', 'anchor_embeddedness',
    'candidate_hs4', 'candidate_product_name_short', 'candidate_sector', 'candidate_hs_section_name',
    'proximity', 'proximity_above_country_median', 'proximity_rank', 'eligible_candidate_count',
    'pci', 'cog', 'distance_travelled', 'accessible_market_size', 'accessible_market_size_share',
    'accessible_market_growth_5y', 'dai_index', 'dai_percentile', 'dai_lead',
    'pci_norm', 'cog_norm', 'accessible_market_size_norm', 'accessible_market_growth_norm',
    'dai_norm', 'attractiveness_score', 'feasibility_score', 'combined_score',
]
panel = panel[anchor_output_cols]
panel.to_csv(OUTPUT / 'anchors_proximity_percentile.csv', index=False)


print('avg complexity rows', len(complexity_avg))
print('avg proximity rows', len(prox_avg))
print('presence rows', len(presence), 'anchors', int(presence['presence'].sum()))
print('anchor-candidate links', len(panel), 'unique candidates', panel['candidate_hs4'].nunique())
panel[['anchor_hs4', 'candidate_hs4', 'proximity', 'dai_percentile', 'accessible_market_size']].head()


2020_2024_avg


Percentage of pairs compared that meet log-supermodularity condition: 58.37%
2020_2024_avg


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ecomplexity/ecomplexity.py:252: UserWarning: Year 2020_2024_avg: Log-supermodularity condition is not fully satisfied (58.37% of pairs compared satisfy this condition). The ECI and PCI values may not be a true representation of the complexity. More details at: https://growthlab.hks.harvard.edu/publications/structural-ranking-economic-complexity
  warnings.warn(


avg complexity rows 179945
avg proximity rows 1483524
presence rows 1241 anchors 153
anchor-candidate links 1683 unique candidates 533


,anchor_hs4,candidate_hs4,proximity,dai_percentile,accessible_market_size
0,0101,3504,0.353508,48.387097,1.827365e+08
1,0101,1506,0.324439,12.903226,3.689364e+06
2,0101,0204,0.317188,6.666667,1.873563e+09
3,0101,1603,0.316552,6.451613,1.126167e+06
4,0101,5905,0.311367,6.451613,8.662450e+05


## 8. Validation


In [10]:
required_outputs = [
    INTERMEDIATE / 'countries.csv',
    INTERMEDIATE / 'trade_hs4_baci_2020_2024.csv',
    INTERMEDIATE / 'hs92_attributes.csv',
    INTERMEDIATE / 'accessible_market_by_product.csv',
    INTERMEDIATE / 'accessible_market_by_product_year.csv',
    INTERMEDIATE / 'accessible_market_growth_by_product.csv',
    INTERMEDIATE / 'complexity_calculations.csv',
    INTERMEDIATE / 'complexity_arg_2024.csv',
    INTERMEDIATE / 'proximity.csv',
    INTERMEDIATE / 'opportunity_metrics_hs4_arg.csv',
    INTERMEDIATE / 'complexity_calculations_avg.csv',
    INTERMEDIATE / 'proximity_avg.csv',
    INTERMEDIATE / f'{FOCUS_COUNTRY_SLUG}_presence.csv',
    INTERMEDIATE / 'anchors.csv',
    OUTPUT / 'anchors_proximity_percentile.csv',
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required outputs: ' + '; '.join(missing))

metrics_check = pd.read_csv(INTERMEDIATE / 'opportunity_metrics_hs4_arg.csv')
complexity_check = pd.read_csv(INTERMEDIATE / 'complexity_arg_2024.csv')
print('metrics rows', len(metrics_check))
print('complexity ARG 2024 rows', len(complexity_check))
print('accessible market total B USD', metrics_check['accessible_market_size'].sum() / 1_000_000_000)
print('DAI percentile range', metrics_check['dai_percentile'].min(), metrics_check['dai_percentile'].max())
print('density percentile range', complexity_check['density_percentile'].min(), complexity_check['density_percentile'].max())
anchor_links_check = pd.read_csv(OUTPUT / 'anchors_proximity_percentile.csv')
anchors_check = pd.read_csv(INTERMEDIATE / 'anchors.csv')
print('anchor rows', len(anchors_check))
print('anchor candidate links rows', len(anchor_links_check))
print('anchor unique candidates', anchor_links_check['candidate_hs4'].nunique())
print('sample anchor candidates')
print(anchor_links_check[['anchor_hs4', 'candidate_hs4', 'proximity', 'dai_percentile', 'accessible_market_size']].head().to_string(index=False))
metrics_check[['hs4', 'accessible_market_size', 'accessible_market_growth_5y', 'dai_index', 'dai_percentile', 'dai_lead']].head()

metrics rows 1241
complexity ARG 2024 rows 1241
accessible market total B USD 1425.434109858
DAI percentile range 3.225806451612903 100.0
density percentile range 0.0 0.7412587412587412
anchor rows 153
anchor candidate links rows 1683
anchor unique candidates 533
sample anchor candidates
 anchor_hs4  candidate_hs4  proximity  dai_percentile  accessible_market_size
        101           3504   0.353508       48.387097             182736489.0
        101           1506   0.324439       12.903226               3689364.0
        101            204   0.317188        6.666667            1873563013.0
        101           1603   0.316552        6.451613               1126167.0
        101           5905   0.311367        6.451613                866245.0


,hs4,accessible_market_size,accessible_market_growth_5y,dai_index,dai_percentile,dai_lead
0,101,20520831.0,0.170474,0.554408,10.000000,-66.666667
1,102,2410793.0,-0.004299,0.738540,12.903226,-61.290323
2,103,164128.0,0.184778,0.443042,6.451613,-41.935484
3,104,121882.0,-0.066684,1.342093,64.516129,-12.903226
4,105,26123495.0,0.038387,1.047235,29.032258,-22.580645
